In [4]:
import os, sys
from pathlib import Path

p = Path.cwd()
while not (p / "src").is_dir() and p != p.parent:
    p = p.parent
os.chdir(p)
sys.path.insert(0, str(p / "src"))
print("repo root:", os.getcwd())

repo root: C:\Users\sudar\Desktop\Preparation\tier-a-prep\precrash-eval


In [7]:
import subprocess, sys

subprocess.run([sys.executable, "scripts/make_synthetic_features.py",
                "--manifest", "data/nexar_fixed.dev.csv",
                "--out_dir", "features/synth", "--limit", "120"])

r = subprocess.run([sys.executable, "scripts/protocol_comparison.py",
                    "--features", "features/synth",
                    "--config", "configs/honest.yaml",
                    "--out", "results/synth.json",
                    "--dataset_name", "SYNTHETIC"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-500:])

SYNTHETIC: 65 positive, 55 negative clips, per-clip rates 29.10-31.00, median 30.00

  decision windows: positives end at their onset; negatives matched to 135-135 frames (unmatched would be 150)

method                     blind     t_c    TTA*   STTA     | AP     AUC  TTA@R80  rank L->C
--------------------------------------------------------------------------------------------
prior_constant_0.51          yes     0.0    4.98   1.00   0.5417  0.5000      nan     1 -> 11 
prior_sigmoid_m0.40          yes    60.0    2.99   1.00   0.5417  0.5000     0.36     2 -> 5  
prior_sigmoid_m0.50          yes    75.0    2.49   1.00   0.5417  0.5000     0.20     3 -> 6  
prior_linear_ramp            yes    75.0    2.49   1.00   0.5417  0.5000     0.03     4 -> 9  
prior_step_at_half           yes    75.0    2.49   1.00   0.5417  0.5000     1.99     5 -> 10 
prior_sigmoid_m0.60          yes    90.0    1.99   1.00   0.5417  0.5000     0.03     6 -> 7  
prior_sigmoid_m0.70          yes   105.0    1.4

In [8]:
try:
    import torch
    print(torch.__version__, "| cuda:", torch.cuda.is_available())
except ImportError:
    print("no torch installed")

no torch installed


In [9]:
subprocess.run([sys.executable, "scripts/analyse_taa.py",
                "--test_csv", "data/test.csv",
                "--submission_csv", "data/sample_submission.csv",
                "--train_csv", "data/train.csv",
                "--reported_crossover", "22.7",
                "--out", "results/taa_analysis.json"])

CompletedProcess(args=['C:\\Users\\sudar\\Desktop\\Preparation\\tier-a-prep\\precrash-eval\\.venv\\Scripts\\python.exe', 'scripts/analyse_taa.py', '--test_csv', 'data/test.csv', '--submission_csv', 'data/sample_submission.csv', '--train_csv', 'data/train.csv', '--reported_crossover', '22.7', '--out', 'results/taa_analysis.json'], returncode=1)

In [12]:
import subprocess, sys, os
from pathlib import Path

for f in ("test.csv", "sample_submission.csv", "train.csv"):
    p = Path("data") / f
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

r = subprocess.run([sys.executable, "scripts/analyse_taa.py",
                    "--test_csv", "data/test.csv",
                    "--submission_csv", "data/sample_submission.csv",
                    "--train_csv", "data/train.csv",
                    "--reported_crossover", "22.7",
                    "--out", "results/taa_analysis.json"],
                   capture_output=True, text=True)
print(r.stdout)
print("--- stderr ---", r.stderr[-1200:])

OK   data\test.csv
OK   data\sample_submission.csv
OK   data\train.csv
data/test.csv: 1417 clips
columns: ['id', 'video_id', 'start_frame', 'end_frame', 'caption']

WHAT THE CORPUS DOES NOT PUBLISH
  label        ABSENT                 -> average precision, area under the ROC curve, false-positive rate
  event_time   ABSENT                 -> time-to-accident measured to the collision
  alert_time   ABSENT                 -> a ceiling to check a reported warning time against

  train.csv: 1 row(s)
    note: This is a zero-shot traffic accident anticipation task. There is no train split. Please use test_kaggle.csv as the only public data for inference.

  window spans: {150: 1417}

THE ORGANISERS' REFERENCE SUBMISSION
  entries              : 1417
  length               : 150
  first, last          : 0.001000, 0.999000
  identical every clip : True
  max deviation from a straight line: 5.10e-07

  One curve, repeated for every clip. It reads no pixels.

CROSSING FRAME OF A VIDEO-BLIND F

In [11]:
import os, subprocess, sys
from getpass import getpass

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"])
os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle token (KGAT_...): ").strip()

os.makedirs("data", exist_ok=True)
for name in ("test.csv", "sample_submission.csv", "train.csv"):
    r = subprocess.run([sys.executable, "-m", "kaggle", "competitions", "download",
                        "-c", "zero-shot-taa", "-f", name, "-p", "data"],
                       capture_output=True, text=True)
    print(name, "->", r.returncode, r.stderr[-200:] if r.returncode else "")

import zipfile, glob
for z in glob.glob("data/*.zip"):
    with zipfile.ZipFile(z) as f: f.extractall("data")
    os.remove(z)

print(sorted(os.listdir("data")))

Kaggle token (KGAT_...):  ········


test.csv -> 0 
sample_submission.csv -> 0 
train.csv -> 0 
['nexar_fixed.csv', 'nexar_fixed.dev.csv', 'nexar_fixed.test.csv', 'nexar_random.csv', 'nexar_random.dev.csv', 'nexar_random.test.csv', 'sample_submission.csv', 'test.csv', 'train.csv']


In [13]:
subprocess.run([sys.executable, "scripts/make_submission.py",
                "--test_csv", "data/test.csv",
                "--sample_csv", "data/sample_submission.csv",
                "--replicate_sample",
                "--out", "submissions/00_replicate_sample.csv"],
               capture_output=True, text=True).stdout

"test.csv           : 1417 ids\nsample_submission  : 1417 ids, 150 values, bracketed=True, sep=','\n\n  format check against sample_submission.csv\n    id sets identical : True\n    risk strings exact: 0 of 1417\n    -> differs, e.g. 1_009334_14_164\n       theirs: [0.001,0.007698,0.014396,0.021094,0.027792,0.03449,0.041188,0.047886,0\n       ours  : [0.001,0.007698,0.014396,0.021094,0.027792,0.03449,0.041188,0.047886,0\n\nwrote submissions\\00_replicate_sample.csv\n  contents      : replicate_sample (known score: 0.58333)\n  rows          : 1417\n  identical     : True\n  first curve   : 0.0010 .. 0.9990, crossing frame 75\n  size          : 1.9 MB\n\nUpload it, then record the score in submissions/LOG.md.\n"

In [14]:
for n, curve in [("01_constant", "constant_0.51"),
                 ("02_linear_ramp", "linear_ramp"),
                 ("03_sigmoid_m050", "sigmoid_m0.50"),
                 ("04_step", "step_at_half"),
                 ("05_sigmoid_m040", "sigmoid_m0.40")]:
    subprocess.run([sys.executable, "scripts/make_submission.py",
                    "--test_csv", "data/test.csv",
                    "--sample_csv", "data/sample_submission.csv",
                    "--curve", curve, "--out", f"submissions/{n}.csv"])

In [15]:
r = subprocess.run([sys.executable, "scripts/make_submission.py",
                    "--test_csv", "data/test.csv",
                    "--sample_csv", "data/sample_submission.csv",
                    "--replicate_sample",
                    "--out", "submissions/00_replicate_sample.csv"],
                   capture_output=True, text=True)
print(r.stdout)

test.csv           : 1417 ids
sample_submission  : 1417 ids, 150 values, bracketed=True, sep=','
float style        : repr

  format check against sample_submission.csv
    id sets identical : True
    risk strings exact: 1417 of 1417
    -> byte-identical. Format confirmed without using a submission slot; this file would score 0.58333.

wrote submissions\00_replicate_sample.csv
  contents      : replicate_sample (known score: 0.58333)
  rows          : 1417
  identical     : True
  first curve   : 0.0010 .. 0.9990, crossing frame 75
  size          : 1.9 MB

Upload it, then record the score in submissions/LOG.md.



In [16]:
import subprocess, sys

curves = [("01_constant",      "constant_0.51"),
          ("02_linear_ramp",   "linear_ramp"),
          ("03_sigmoid_m050",  "sigmoid_m0.50"),
          ("04_step_at_half",  "step_at_half"),
          ("05_sigmoid_m040",  "sigmoid_m0.40")]

for name, curve in curves:
    r = subprocess.run([sys.executable, "scripts/make_submission.py",
                        "--test_csv", "data/test.csv",
                        "--sample_csv", "data/sample_submission.csv",
                        "--curve", curve,
                        "--out", f"submissions/{name}.csv"],
                       capture_output=True, text=True)
    tail = [l for l in r.stdout.splitlines() if "crossing" in l or "size" in l]
    print(f"{name:20} {'OK' if r.returncode == 0 else 'FAILED'}  {' '.join(tail)}")
    if r.returncode: print(r.stderr[-400:])

01_constant          OK    first curve   : 0.5100 .. 0.5100, crossing frame 0   size          : 1.1 MB
02_linear_ramp       OK    first curve   : 0.0000 .. 1.0000, crossing frame 75   size          : 4.1 MB
03_sigmoid_m050      OK    first curve   : 0.0025 .. 0.9975, crossing frame 75   size          : 4.2 MB
04_step_at_half      OK    first curve   : 0.0050 .. 0.9950, crossing frame 75   size          : 1.3 MB
05_sigmoid_m040      OK    first curve   : 0.0082 .. 0.9993, crossing frame 60   size          : 4.1 MB


In [17]:
import subprocess, sys, os, time
from getpass import getpass

if "KAGGLE_API_TOKEN" not in os.environ:
    os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle token (KGAT_...): ").strip()

SUBS = [
    ("01_constant.csv",     "constant 0.51 - video-blind control, reads no pixels"),
    ("02_linear_ramp.csv",  "linear ramp 0 to 1 - video-blind"),
    ("03_sigmoid_m050.csv", "sigmoid midpoint 0.50 - video-blind"),
    ("04_step_at_half.csv", "step at frame 75 - video-blind"),
    ("05_sigmoid_m040.csv", "sigmoid midpoint 0.40 - video-blind"),
]

for fname, msg in SUBS:
    r = subprocess.run([sys.executable, "-m", "kaggle", "competitions", "submit",
                        "-c", "zero-shot-taa",
                        "-f", f"submissions/{fname}",
                        "-m", msg],
                       capture_output=True, text=True)
    print(f"{fname:22} {'OK' if r.returncode == 0 else 'FAILED'}  "
          f"{r.stdout.strip()[-70:]}{r.stderr.strip()[-200:] if r.returncode else ''}")
    time.sleep(3)

01_constant.csv        OK  ining today.
Successfully submitted to Zero-shot Accident Anticipation
02_linear_ramp.csv     OK  ining today.
Successfully submitted to Zero-shot Accident Anticipation
03_sigmoid_m050.csv    OK  ining today.
Successfully submitted to Zero-shot Accident Anticipation
04_step_at_half.csv    OK  ining today.
Successfully submitted to Zero-shot Accident Anticipation
05_sigmoid_m040.csv    OK  ining today.
Successfully submitted to Zero-shot Accident Anticipation


In [19]:
r = subprocess.run([sys.executable, "-m", "kaggle", "competitions", "submissions",
                    "-c", "zero-shot-taa"], capture_output=True, text=True)
print(r.stdout)

     ref  fileName             date                        description                                           status                     publicScore  privateScore  
--------  -------------------  --------------------------  ----------------------------------------------------  -------------------------  -----------  ------------  
55907867  05_sigmoid_m040.csv  2026-08-31 05:17:14.190000  sigmoid midpoint 0.40 - video-blind                   SubmissionStatus.COMPLETE  1.35057      1.35428       
55907863  04_step_at_half.csv  2026-08-31 05:17:00.743000  step at frame 75 - video-blind                        SubmissionStatus.COMPLETE  1.10057      1.10442       
55907861  03_sigmoid_m050.csv  2026-08-31 05:16:49.990000  sigmoid midpoint 0.50 - video-blind                   SubmissionStatus.COMPLETE  1.10057      1.10435       
55907856  02_linear_ramp.csv   2026-08-31 05:16:37.207000  linear ramp 0 to 1 - video-blind                      SubmissionStatus.COMPLETE  1.06604      1.06725

In [20]:
import os
print(os.getcwd())          # should end in \precrash-eval
print("probe" in open("scripts/make_submission.py").read())   # should be True

C:\Users\sudar\Desktop\Preparation\tier-a-prep\precrash-eval
True


In [21]:
import subprocess, sys, os, glob, time
from getpass import getpass

if "KAGGLE_API_TOKEN" not in os.environ:
    os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle token (KGAT_...): ").strip()

PROBES = ["never_crosses", "cross_then_drop", "constant_0.51", "constant_0.99",
          "cross_then_dip", "step_at_000", "step_at_010", "step_at_025",
          "step_at_050", "step_at_075", "step_at_100", "step_at_125",
          "step_at_140"]
CURVES = ["sigmoid_m0.60", "sigmoid_m0.70"]

# ---- generate ----
for kind, names in (("--probe", PROBES), ("--curve", CURVES)):
    for n in names:
        tag = "p_" if kind == "--probe" else "c_"
        r = subprocess.run([sys.executable, "scripts/make_submission.py",
                            "--test_csv", "data/test.csv",
                            "--sample_csv", "data/sample_submission.csv",
                            kind, n, "--out", f"submissions/{tag}{n}.csv"],
                           capture_output=True, text=True)
        if r.returncode:
            print("GEN FAILED", n, r.stderr[-300:])
print("generated:", len(glob.glob("submissions/p_*.csv") + glob.glob("submissions/c_*.csv")))

# ---- submit ----
files = sorted(glob.glob("submissions/p_*.csv")) + sorted(glob.glob("submissions/c_*.csv"))
for f in files:
    name = os.path.basename(f)[:-4]
    r = subprocess.run([sys.executable, "-m", "kaggle", "competitions", "submit",
                        "-c", "zero-shot-taa", "-f", f, "-m", f"probe {name}"],
                       capture_output=True, text=True)
    print(f"  {name:24} {'OK' if r.returncode == 0 else 'FAIL ' + r.stderr[-150:]}")
    time.sleep(3)

generated: 15
  p_constant_0.51          OK
  p_constant_0.99          OK
  p_cross_then_dip         OK
  p_cross_then_drop        OK
  p_never_crosses          OK
  p_step_at_000            OK
  p_step_at_010            OK
  p_step_at_025            OK
  p_step_at_050            OK
  p_step_at_075            OK
  p_step_at_100            OK
  p_step_at_125            OK
  p_step_at_140            OK
  c_sigmoid_m0.60          OK
  c_sigmoid_m0.70          OK


In [22]:
r = subprocess.run([sys.executable, "-m", "kaggle", "competitions", "submissions",
                    "-c", "zero-shot-taa", "-v"], capture_output=True, text=True)
open("submissions/scores.csv", "w", encoding="utf-8").write(r.stdout)
print(r.stdout)

ref,fileName,date,description,status,publicScore,privateScore

55908070,c_sigmoid_m0.70.csv,2026-08-31 05:29:10.353000,probe c_sigmoid_m0.70,SubmissionStatus.COMPLETE,0.55190,0.55763

55908065,c_sigmoid_m0.60.csv,2026-08-31 05:28:57.810000,probe c_sigmoid_m0.60,SubmissionStatus.COMPLETE,0.84398,0.84840

55908061,p_step_at_140.csv,2026-08-31 05:28:43.647000,probe p_step_at_140,SubmissionStatus.COMPLETE,0.35088,0.35105

55908057,p_step_at_125.csv,2026-08-31 05:28:31.043000,probe p_step_at_125,SubmissionStatus.COMPLETE,0.37383,0.37967

55908054,p_step_at_100.csv,2026-08-31 05:28:17.820000,probe p_step_at_100,SubmissionStatus.COMPLETE,0.68391,0.68822

55908050,p_step_at_075.csv,2026-08-31 05:28:06.123000,probe p_step_at_075,SubmissionStatus.COMPLETE,1.10057,1.10445

55908043,p_step_at_050.csv,2026-08-31 05:27:54.290000,probe p_step_at_050,SubmissionStatus.COMPLETE,1.51724,1.52088

55908039,p_step_at_025.csv,2026-08-31 05:27:41.357000,probe p_step_at_025,SubmissionStatus.COMPLETE,1.93391,1.